<a href="https://www.kaggle.com/code/kamifiza/06-mlops-tracking-with-wandb?scriptVersionId=289153113" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# MLOps with Weights & Biases for Association Rule Mining
# Instacart Market Basket Analysis

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/03-apriori-experiments/__results__.html
/kaggle/input/03-apriori-experiments/__notebook__.ipynb
/kaggle/input/03-apriori-experiments/__output__.json
/kaggle/input/03-apriori-experiments/custom.css
/kaggle/input/03-apriori-experiments/apriori_results_instacart/apriori_b72a1357_sup0_002_conf0_2_lift1_0.csv
/kaggle/input/03-apriori-experiments/apriori_results_instacart/apriori_cc18c6da_sup0_001_conf0_2_lift1_0.csv
/kaggle/input/03-apriori-experiments/apriori_results_instacart/apriori_5702436d_sup0_001_conf0_1_lift1_0.csv
/kaggle/input/03-apriori-experiments/apriori_results_instacart/apriori_c16e86e2_sup0_002_conf0_1_lift1_0.csv
/kaggle/input/03-apriori-experiments/apriori_results_instacart/apriori_376748ce_sup0_002_conf0_3_lift1_0.csv
/kaggle/input/03-apriori-experiments/apriori_results_instacart/apriori_cc4fe19a_sup0_01_conf0_1_lift1_0.csv
/kaggle/input/03-apriori-experiments/apriori_results_instacart/apriori_ddd0a667_sup0_01_conf0_3_lift1_0.csv
/kaggle/input/03-apriori-exp

### Installation of WNDB

In [2]:
!pip install wandb


### Login to wndb

In [3]:
import wandb
wandb.login(key = "53d90faf2d4a96a8896be16279eb43067cfdfe2c")


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

True

### 1-  W&B Project Initialization
his notebook does not re-train models.  
Instead, it **logs existing FP-Growth and Apriori experiments** to W&B, versions the preprocessings 

In [4]:
import wandb
import os

PROJECT_NAME = "instacart-market-basket-mlops"

# login W&B
wandb.login(key=os.getenv("WANDB_API_KEY"))

# init run
run = wandb.init(project=PROJECT_NAME, job_type="data_preparation")

# chemins fichiers
matrix_path = "/kaggle/input/01-preprocessing-and-eda/processed_instacart/instacart_transaction_matrix.parquet"
transactions_path = "/kaggle/input/01-preprocessing-and-eda/processed_instacart/instacart_transactions_filtered.pkl"

# créer un artifact dataset
data_art = wandb.Artifact(
    name="instacart-transactions-prepared",
    type="dataset",
    metadata={"description": "Filtered transactions + boolean matrix for association rules"}
)

# ajouter les fichiers
data_art.add_file(matrix_path)
data_art.add_file(transactions_path)

# log artifact
run.log_artifact(data_art, aliases=["v0", "baseline"])
run.finish()
print("✔ Dataset artifact logged!")


✔ Dataset artifact logged!


### 2- Load Offline Experiment Results 
We load the outputs produced by previous notebooks:- FP-Growth experiments summary  - Apriori experiments summary  - Best model rules and evaluation summary  - Preprocessed transaction matrix (dataset) 

In [5]:
# ============================================================
# PHASE 2 : Charger les données et résumés des expériences
# ============================================================
import os
import glob
import pandas as pd

# -----------------------------
# 1️⃣ Définir les chemins Kaggle
# -----------------------------
DATA_DIR = "/kaggle/input/01-preprocessing-and-eda/processed_instacart"
transaction_matrix_path = os.path.join(DATA_DIR, "instacart_transaction_matrix.parquet")
transactions_filtered_path = os.path.join(DATA_DIR, "instacart_transactions_filtered.pkl")

FP_DIR = "/kaggle/input/02-fp-growth-experiments/fp_growth_results_instacart"
fp_files = glob.glob(os.path.join(FP_DIR, "*.csv"))

AP_DIR = "/kaggle/input/03-apriori-experiments/apriori_results_instacart"
ap_files = glob.glob(os.path.join(AP_DIR, "*.csv"))

BEST_DIR = "/kaggle/input/04-best-model-selection/best_association_model"
best_rules_path = os.path.join(BEST_DIR, "best_model_rules_apriori.csv")
best_eval_path = os.path.join(BEST_DIR, "model_selection_summary.csv")

# -----------------------------
# 2️⃣ Charger les datasets
# -----------------------------
df_matrix = pd.read_parquet(transaction_matrix_path)
df_transactions = pd.read_pickle(transactions_filtered_path)
print("Transaction matrix shape:", df_matrix.shape)
print("Filtered transactions shape:", df_transactions.shape)

# -----------------------------
# 3️⃣ Concaténer tous les CSV FP-Growth
# -----------------------------
fp_summary = pd.concat([pd.read_csv(f) for f in fp_files], ignore_index=True) if fp_files else pd.DataFrame()
print("FP-Growth summary shape:", fp_summary.shape)

# -----------------------------
# 4️⃣ Concaténer tous les CSV Apriori
# -----------------------------
ap_summary = pd.concat([pd.read_csv(f) for f in ap_files], ignore_index=True) if ap_files else pd.DataFrame()
print("Apriori summary shape:", ap_summary.shape)

# -----------------------------
# 5️⃣ Charger le best model
# -----------------------------
best_rules = pd.read_csv(best_rules_path) if os.path.exists(best_rules_path) else pd.DataFrame()
best_eval = pd.read_csv(best_eval_path) if os.path.exists(best_eval_path) else pd.DataFrame()
print("Best model rules shape:", best_rules.shape)
print("Best model evaluation shape:", best_eval.shape)


Transaction matrix shape: (50000, 8290)
Filtered transactions shape: (2991160, 2)
FP-Growth summary shape: (410, 29)
Apriori summary shape: (8338, 26)
Best model rules shape: (54, 21)
Best model evaluation shape: (16, 18)


### 3-  Helper to Log a Single Run to W&B

For each experiment (one row in the summary CSV) we:- Initialize a W&B run with config (algorithm + hyperparameters)  - Log metrics (runtime, counts, average confidence, average lift)  - Attach the rule CSV and the preprocessed transaction matrix as artifacts when available
This is analogous to the per-config runs in your SEDS lab.


In [6]:
import wandb
import math
import os

# ============================================================
# 1️⃣ Configuration W&B
# ============================================================
PROJECT_NAME = "instacart-market-basket-mlops"
ENTITY_NAME = "ak-fezazi-university"  # mettre ton W&B username

# ============================================================
# 2️⃣ Fonctions sécurisées pour conversion
# ============================================================
def safe_int(x, default=0):
    try:
        return int(x)
    except (ValueError, TypeError):
        return default

def safe_float(x, default=0.0):
    try:
        return float(x)
    except (ValueError, TypeError):
        return default

# ============================================================
# 3️⃣ Filtrer les expériences valides
# ============================================================
# Vérifier les chemins de fichiers et créer des DataFrames valides
fp_summary_valid = fp_summary.copy() if not fp_summary.empty else pd.DataFrame()
ap_summary_valid = ap_summary.copy() if not ap_summary.empty else pd.DataFrame()

# Ajouter une colonne pour vérifier l'existence des fichiers
if not fp_summary_valid.empty and 'rules_path' in fp_summary_valid.columns:
    fp_summary_valid['file_exists'] = fp_summary_valid['rules_path'].apply(
        lambda x: os.path.exists(str(x)) if isinstance(x, str) else False
    )
    # Filtrer seulement les expériences avec fichiers existants
    fp_summary_valid = fp_summary_valid[fp_summary_valid['file_exists']].copy()

if not ap_summary_valid.empty and 'rules_path' in ap_summary_valid.columns:
    ap_summary_valid['file_exists'] = ap_summary_valid['rules_path'].apply(
        lambda x: os.path.exists(str(x)) if isinstance(x, str) else False
    )
    # Filtrer seulement les expériences avec fichiers existants
    ap_summary_valid = ap_summary_valid[ap_summary_valid['file_exists']].copy()

print(f"FP-Growth valides: {len(fp_summary_valid)} expériences")
print(f"Apriori valides: {len(ap_summary_valid)} expériences")

# ============================================================
# 4️⃣ Helper : log une expérience FP-Growth ou Apriori
# ============================================================
def log_association_run(algorithm, row, rules_path=None, matrix_path=None,
                        project=PROJECT_NAME, entity=ENTITY_NAME, job_type="association_run"):

    # Configuration hyperparamètres
    config = {
        "algorithm": algorithm,
        "min_support": safe_float(row.get("min_support", 0.0)),
        "min_confidence": safe_float(row.get("min_confidence", 0.0)),
        "min_lift": safe_float(row.get("min_lift", 1.0)),
    }

    run = wandb.init(project=project, entity=entity, job_type=job_type, config=config, reinit=True)

    # Metrics de l'expérience
    metrics = {
        "runtime_sec": safe_float(row.get("runtime_sec", 0.0)),
        "n_frequent_itemsets": safe_int(row.get("n_frequent_itemsets", 0)),
        "n_rules": safe_int(row.get("n_rules", 0)),
        "avg_confidence": safe_float(row.get("avg_confidence", 0.0)),
        "avg_lift": safe_float(row.get("avg_lift", 0.0)),
        "max_lift": safe_float(row.get("max_lift", 0.0)),
    }
    wandb.log(metrics)

    # Vérifier rules_path
    if isinstance(rules_path, float) and math.isnan(rules_path):
        rules_path = None

    if rules_path and os.path.exists(rules_path):
        rules_artifact = wandb.Artifact(
            name=f"{algorithm}-rules-{row.get('experiment_id', 'run')}",
            type="rules",
            metadata=config
        )
        rules_artifact.add_file(rules_path)
        run.log_artifact(rules_artifact)

    # Dataset artifact
    if matrix_path and os.path.exists(matrix_path):
        data_artifact = wandb.Artifact(
            name="instacart-transaction-matrix",
            type="dataset",
            metadata={"source": "preprocessed_instacart"}
        )
        data_artifact.add_file(matrix_path)
        run.log_artifact(data_artifact)

    run.finish()

# ============================================================
# 5️⃣ Log FP-Growth experiments valides
# ============================================================
print("\n" + "="*50)
print("Logging FP-Growth experiments...")
print("="*50)

if not fp_summary_valid.empty:
    for idx, row in fp_summary_valid.iterrows():
        experiment_id = row.get('experiment_id', f'fp_{idx}')
        rules_path = row.get("rules_path")
        print(f"Logging FP-Growth experiment {experiment_id}...")
        
        try:
            log_association_run("FP-Growth", row, rules_path=rules_path, matrix_path=transaction_matrix_path)
            print(f"✅ Experiment {experiment_id} logged successfully")
        except Exception as e:
            print(f"❌ Error logging experiment {experiment_id}: {e}")
else:
    print("No valid FP-Growth experiments to log")

# ============================================================
# 6️⃣ Log Apriori experiments valides
# ============================================================
print("\n" + "="*50)
print("Logging Apriori experiments...")
print("="*50)

if not ap_summary_valid.empty:
    for idx, row in ap_summary_valid.iterrows():
        experiment_id = row.get('experiment_id', f'ap_{idx}')
        rules_path = row.get("rules_path")
        print(f"Logging Apriori experiment {experiment_id}...")
        
        try:
            log_association_run("Apriori", row, rules_path=rules_path, matrix_path=transaction_matrix_path)
            print(f"✅ Experiment {experiment_id} logged successfully")
        except Exception as e:
            print(f"❌ Error logging experiment {experiment_id}: {e}")
else:
    print("No valid Apriori experiments to log")

# ============================================================
# 7️⃣ Log du best model
# ============================================================
print("\n" + "="*50)
print("Logging best model...")
print("="*50)

artifact = wandb.Artifact(
    name="best_association_model",
    type="dataset",
    description="Best rules selected from FP-Growth / Apriori experiments"
)

if not best_rules.empty:
    best_rules.to_csv("best_rules.csv", index=False)
    artifact.add_file("best_rules.csv")
    print("✅ Best rules added to artifact")

if not best_eval.empty:
    best_eval.to_csv("evaluation_summary.csv", index=False)
    artifact.add_file("evaluation_summary.csv")
    print("✅ Evaluation summary added to artifact")

try:
    run = wandb.init(project=PROJECT_NAME, entity=ENTITY_NAME, job_type="best_model")
    run.log_artifact(artifact)
    run.finish()
    print("✅ Best model logged successfully")
except Exception as e:
    print(f"❌ Error logging best model: {e}")

print("\n✔ Phase 3 complète : toutes les expériences FP-Growth, Apriori et le best model ont été loggés sur W&B !")

FP-Growth valides: 0 expériences
Apriori valides: 0 expériences

Logging FP-Growth experiments...
No valid FP-Growth experiments to log

Logging Apriori experiments...
No valid Apriori experiments to log

Logging best model...
✅ Best rules added to artifact
✅ Evaluation summary added to artifact


✅ Best model logged successfully

✔ Phase 3 complète : toutes les expériences FP-Growth, Apriori et le best model ont été loggés sur W&B !
